In [3]:
# !pip install cdsapi

In [3]:
import cdsapi

dataset = "reanalysis-era5-pressure-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": ["temperature"],
    "year": ["2024"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "06:00", "12:00",
        "18:00"
    ],
    "pressure_level": ["925", "1000"],
    "data_format": "grib",
    "download_format": "unarchived",
    "area": [53.55394, 14.42802, 53.30394, 14.67802]
}

client = cdsapi.Client(
    url = "https://cds.climate.copernicus.eu/api",
    key = "f55a5040-c43c-40a7-957a-f078b44210f3"
)
client.retrieve(dataset, request).download()


2026-04-04 16:11:19,530 INFO Request ID is 829cf791-27c5-46c5-9592-1f1b3d26ea19
2026-04-04 16:11:19,732 INFO status has been updated to accepted
2026-04-04 16:11:33,652 INFO status has been updated to running
2026-04-04 16:11:41,401 INFO status has been updated to successful


4fd450d6e602b76d4d81bd5b3bd73b04.grib:   0%|          | 0.00/315k [00:00<?, ?B/s]

'4fd450d6e602b76d4d81bd5b3bd73b04.grib'

In [4]:
# !pip install cfgrib eccodes

In [27]:
# !pip install eccodes

In [3]:
!conda install -c conda-forge eccodes cfgrib xarray -y

Channels:
 - conda-forge
 - defaults
Platform: osx-64
Solving environment: done

## Package Plan ##

  environment location: /opt/anaconda3

  added / updated specs:
    - cfgrib
    - eccodes
    - xarray


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    blosc-1.21.5               |       hafa3907_1          46 KB  conda-forge
    bottleneck-1.6.0           |np2py312he8eb05d_3         153 KB  conda-forge
    ca-certificates-2026.2.25  |       hbd8a1cb_0         144 KB  conda-forge
    certifi-2026.2.25          |     pyhd8ed1ab_0         148 KB  conda-forge
    cfgrib-0.9.15.1            |     pyhd8ed1ab_0          43 KB  conda-forge
    conda-24.11.3              |  py312hb401068_0         1.1 MB  conda-forge
    eccodes-2.35.0             |       h40b907c_0         4.3 MB  conda-forge
    findlibs-0.1.2             |     pyhd8ed1ab_0          16 KB  conda-forge
    h5py-3.13.0        

In [5]:
import eccodes
import xarray as xr
import cfgrib

# Odczyt pliku GRIB przez xarray
ds = xr.open_dataset(
    '4fd450d6e602b76d4d81bd5b3bd73b04.grib',
    engine='cfgrib'
)

print(ds)

Ignoring index file '4fd450d6e602b76d4d81bd5b3bd73b04.grib.5b7b6.idx' older than GRIB file


<xarray.Dataset> Size: 35kB
Dimensions:        (time: 1464, isobaricInhPa: 2, latitude: 1, longitude: 1)
Coordinates:
  * time           (time) datetime64[ns] 12kB 2024-01-01 ... 2024-12-31T18:00:00
    valid_time     (time) datetime64[ns] 12kB ...
  * isobaricInhPa  (isobaricInhPa) float64 16B 1e+03 925.0
  * latitude       (latitude) float64 8B 53.5
  * longitude      (longitude) float64 8B 14.5
    number         int64 8B ...
    step           timedelta64[ns] 8B ...
Data variables:
    t              (time, isobaricInhPa, latitude, longitude) float32 12kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-04-04T16:11 GRIB to CDM+CF via cfgrib-0.9.1...


In [6]:
import pandas as pd

df = ds.to_dataframe().reset_index()


# Wyświetl wszystkie wiersze i kolumny
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(df.shape)  # ile wierszy x kolumn
df.head()

(2928, 8)


,time,isobaricInhPa,latitude,longitude,number,step,valid_time,t
0,2024-01-01 00:00:00,1000.0,53.5,14.5,0,0 days,2024-01-01 00:00:00,278.467041
1,2024-01-01 00:00:00,925.0,53.5,14.5,0,0 days,2024-01-01 00:00:00,277.290039
2,2024-01-01 06:00:00,1000.0,53.5,14.5,0,0 days,2024-01-01 06:00:00,277.591797
3,2024-01-01 06:00:00,925.0,53.5,14.5,0,0 days,2024-01-01 06:00:00,276.446045
4,2024-01-01 12:00:00,1000.0,53.5,14.5,0,0 days,2024-01-01 12:00:00,279.575928


In [13]:
# Rozdzielam na dwa DataFrame 
df_925 = df[df['isobaricInhPa'] == 925][['time','t']].rename(columns = {'t': 't_925'})
df_1000 = df[df['isobaricInhPa'] == 1000][['time','t']].rename(columns = {'t': 't_1000'})

# Łaczymy oba df, w jeden do obliczenia różnicy
df_delta = pd.merge(df_925, df_1000, how='inner', on='time')

# kolumna t_delta = podaje rożnice w stopniach 
df_delta['t_delta'] = df_delta['t_1000']-df_delta['t_925']
df_delta['inversion_0/1'] = [1 if e > 0 else 0 for e in df_delta['t_delta']]

# wyciagamy sama date , do łatiwejszegeo grupowania i wyciagniecia dziennej sreniej 

df_delta['data'] = df_delta['time'].dt.date


In [15]:
df_delta.to_csv('/Users/marcinurbanski/Desktop/merito_projekt_pliki_robocze/qualityair_projekt/ready_raw_data/inwersja_szn_2024.csv')

In [9]:
import pandas as pd

# Konwersja do DataFrame (temperatura w Kelvinach -> Celsjusze)
df = ds['t'].to_dataframe().reset_index()
df['temperature_C'] = df['t'] - 273.15

print(df.head(20))
print(f"\nZakres dat: {df['time'].min()} → {df['time'].max()}")
print(f"Poziomy ciśnienia: {df['isobaricInhPa'].unique()}")

                  time  isobaricInhPa  number   step  latitude  longitude  \
0  2024-01-01 00:00:00         1000.0       0 0 days      53.5       14.5   
1  2024-01-01 00:00:00          925.0       0 0 days      53.5       14.5   
2  2024-01-01 06:00:00         1000.0       0 0 days      53.5       14.5   
3  2024-01-01 06:00:00          925.0       0 0 days      53.5       14.5   
4  2024-01-01 12:00:00         1000.0       0 0 days      53.5       14.5   
5  2024-01-01 12:00:00          925.0       0 0 days      53.5       14.5   
6  2024-01-01 18:00:00         1000.0       0 0 days      53.5       14.5   
7  2024-01-01 18:00:00          925.0       0 0 days      53.5       14.5   
8  2024-01-02 00:00:00         1000.0       0 0 days      53.5       14.5   
9  2024-01-02 00:00:00          925.0       0 0 days      53.5       14.5   
10 2024-01-02 06:00:00         1000.0       0 0 days      53.5       14.5   
11 2024-01-02 06:00:00          925.0       0 0 days      53.5       14.5   